# Tool Use and Reflective Agents

In this lab, you will explore how [AI agents](https://class.vision/blog/ai-agents/) can enhance research workflows by leveraging external tools and engaging in critical self-reflection. You'll learn how to build and integrate callable tools—such as web and academic search functions, and connect them to a language model using OpenAI's tool-calling API. Then, you’ll guide the agent to not only generate content but also **reflect** on its own output, improving the quality and depth of the final report. By the end of this lab, you will have implemented a mini agent capable of searching, reasoning, and publishing structured reports in HTML—laying the foundation for more advanced multi-step and autonomous AI systems.

### 🎯 Learning Objectives

By the end of this lab, you can:
- Chain steps into a research pipeline (**search → reflection → formatting**).
- Convert natural-language output into **styled HTML** suitable for sharing.

## ⚙️ Setup

This section:
- Loads environment variables
- Instantiates the OpenAI client

In [ ]:
!pip install -q tavily

In [4]:
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-..."
os.environ["TAVILY_API_KEY"] = "tvly-..." #https://www.tavily.com/

In [ ]:
if 'COLAB_GPU' in os.environ or not os.path.exists('utils'):
    print("📥 Downloading required files...")
    !wget -q https://raw.githubusercontent.com/Alireza-Akhavan/Agentic_AI/refs/heads/main/utils/research_tools.py -P utils
    !pip install -q aisuite
    print("✅ Setup completed")
else:
    print("✅ Running locally - using existing files")

In [5]:
# ================================
# Standard library imports
# ================================
import json
import requests

# ================================
# Third-party imports
# ================================
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML


# ================================
# Local / project imports
# ================================
from utils import research_tools

# ================================
# Environment setup
# ================================
load_dotenv()  # Load environment variables from .env file
client = OpenAI()

## 🧰 Provided Tools

You’ll use two research helpers exposed in `research_tools`:
- **`arxiv_search_tool(query, max_results)`** – academic papers via arXiv API.
- **`tavily_search_tool(query, max_results, include_images)`** – general web search via Tavily.

## 🛠️ `arxiv_search_tool`

Searches arXiv and returns a list of papers with:
- `title`, `authors`, `published`, `summary`, `url`, and (if available) `link_pdf`.

Below, we run a quick test and print the results in a readable format.


In [6]:
# Test the arXiv search tool
results = research_tools.arxiv_search_tool("retrieval-augmented generation", max_results=3)

# Show formatted results
for i, paper in enumerate(results, 1):
    if "error" in paper:
        print(f"❌ Error: {paper['error']}")
    else:
        print(f"📄 Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")


print("\n🧾 Raw Results:\n")
print(json.dumps(results, indent=2))

📄 Paper 1
  Title     : AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
  Authors   : Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang
  Published : 2025-06-08
  URL       : http://arxiv.org/abs/2506.06962v3

📄 Paper 2
  Title     : EVOR: Evolving Retrieval for Code Generation
  Authors   : Hongjin Su, Shuyang Jiang, Yuhang Lai, Haoyuan Wu, Boao Shi, Che Liu, Qian Liu, Tao Yu
  Published : 2024-02-19
  URL       : http://arxiv.org/abs/2402.12317v2

📄 Paper 3
  Title     : Automated Literature Review Using NLP Techniques and LLM-Based Retrieval-Augmented Generation
  Authors   : Nurshat Fateh Ali, Md. Mahdi Mohtasim, Shakil Mosharrof, T. Gopi Krishna
  Published : 2024-11-27
  URL       : http://arxiv.org/abs/2411.18583v1


🧾 Raw Results:

[
  {
    "title": "AR-RAG: Autoregressive Retrieval Augmentation for Image Generation",
    "authors": [
      "Jingyuan Qi",
      "Zhiyang Xu",
      "Qifan Wang",
      "Lifu Huang"
    ],
    "published": "2025-06-08",
    "url"

## 🛠️ `tavily_search_tool`

Calls the Tavily API to fetch web results. Returns a list of dicts:
- `title`, `content`, `url` (and optional image URLs when `include_images=True`).

Run the cell to inspect sample output.

In [8]:
# Test the Tavily search tool
search_results = research_tools.tavily_search_tool("retrieval-augmented generation applications")
for item in search_results:
    print(item)

{'title': 'Retrieval augmented generation use cases', 'content': "* By integrating retrieval mechanisms with language generation, RAG systems produce more accurate and informative text outputs, significantly improving tasks like machine translation, question answering, and summarization. Retrieval augmented generation (RAG) is an artificial intelligence methodology that combines the power of neural language models with external knowledge resources to generate text that is relevant and informed. Retrieval augmented generation (RAG) operates by integrating a retrieval component into the language generation process, expanding the model's knowledge base beyond its initial training data. Retrieval augmented generation (RAG) significantly enhances the capabilities of natural language processing systems. For **question answering**, RAG employs its retrieval component to source relevant information before generating a response. Retrieval-augmented generation (RAG) technology significantly impr

## 🔗 Tool Mapping

We map tool names (strings) to the actual Python functions. This allows the model to call tools by name during tool-calling.

In [9]:
# Tool mapping
tool_mapping = {
    "tavily_search_tool": research_tools.tavily_search_tool,
    "arxiv_search_tool": research_tools.arxiv_search_tool,
}

---

### 🧠 Tool-Calling Research Assistant

`generate_research_report_with_tools(prompt_)` :
1. Builds the conversation with a system message + user prompt.
2. Lets the model **call tools** (`arxiv_search_tool`, `tavily_search_tool`) as needed.
3. Executes tool calls, appends results, and continues the loop until a final answer.
4. Returns the **full message history** (including tool calls and responses).



In [10]:
def generate_research_report_with_tools(prompt_: str, model: str = "gpt-4o") -> str:
    """
    Generates a research report using OpenAI's tool-calling with arXiv and Tavily tools.

    Args:
        prompt_ (str): The user prompt.
        model (str): OpenAI model name.

    Returns:
        str: Final assistant research report text.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "🔍 Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "📚 Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "🌐 When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "✍️ Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "🚫 Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },
        {"role": "user", "content": prompt_}
    ]

    functions = [research_tools.arxiv_tool_def, research_tools.tavily_tool_def]
    MAX_TURNS = 10
    final_text = None

    for _ in range(MAX_TURNS):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=functions,
            tool_choice="auto",
            temperature=1,
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            final_text = msg.content
            print("✅ Final answer:")
            print(final_text)
            break

        for call in msg.tool_calls:
            tool_name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"🛠️ {tool_name}({args})")

            try:
                tool_func = tool_mapping[tool_name]
                result = tool_func(**args)
            except Exception as e:
                result = {"error": str(e)}

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(result)
            })

    return final_text or ""



---

### 🧠 Reflective Research Tool

`reflection(text_or_messages)` produces a short, structured critique with:
- **Strengths**
- **Limitations**
- **Suggestions**
- **Opportunities**

It uses an academic, concise tone. Accept either raw text or the `messages` list from the tool-calling step.

In [14]:
def reflection_and_rewrite(text_or_messages, model: str = "gpt-5.1", temperature: float = 0.3) -> dict:
    """
    Generates a structured reflection AND a revised research report.
    Accepts raw text OR the messages list returned by generate_research_report_with_tools.

    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report
    """
    # Extract assistant content if messages were passed
    if isinstance(text_or_messages, list):
        text = None
        for m in reversed(text_or_messages):
            role = m.get("role") if isinstance(m, dict) else getattr(m, "role", None)
            content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
            if role == "assistant" and content:
                text = content
                break
        if not text:
            raise ValueError("No assistant text found in messages.")
    else:
        text = str(text_or_messages)

    # Ask for both reflection + rewritten report
    user_prompt = (
        "First, provide a structured reflection (Strengths, Limitations, Suggestions, Opportunities) "
        "on the following report.\n\n"
        "Then, write a revised version of the report that incorporates your suggestions, "
        "improves clarity, and strengthens academic tone.\n\n"
        f"Report:\n{text}"
    )

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an academic reviewer and editor."},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )

    # Expect the model to produce both sections in one response
    full_output = resp.choices[0].message.content.strip()

    return {
        "reflection": full_output,   # includes reflection
        "revised_report": full_output  # same output may include both parts
    }



### 🧠 Publish as HTML

`generate_research_report_with_tools(prompt_, model="gpt-5.1")` lets the model **call tools**, gather **evidence with citations**, and produce a **sourced answer**, then convert it to HTML.

**What it does (brief)**
1. Starts a chat with a **system policy** (use tools, cite URLs, be neutral) + your **prompt**.
2. Exposes two tools:
   - `arxiv_search_tool(query, max_results)`
   - `tavily_search_tool(query, max_results, include_images)`
3. Loops up to `MAX_TURNS`: executes tool calls, appends results; **stops** when the assistant replies with no more tool calls.
4. Normalizes messages to plain dicts (`role`, `content`, `tool_calls`) for easy downstream use.
5. Returns the **full message history** (dialogue, calls, results, final answer).

**Args**
- `prompt_` *(str)* – your research question.
- `model` *(str, default `gpt-4o`)*.

**Returns**
- `list` of message dicts including the final sourced answer.

In [15]:
def convert_report_to_html(text_or_messages, model: str = "gpt-5.1") -> str:
    """
    Converts a plaintext research report into a styled HTML page using OpenAI.
    Accepts raw text OR the messages list from the tool-calling step.
    """
    # Inline extraction (sin helper)
    if isinstance(text_or_messages, list):
        text_report = None
        for m in reversed(text_or_messages):
            role = m.get("role") if isinstance(m, dict) else getattr(m, "role", None)
            content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
            if role == "assistant" and content:
                text_report = content
                break
        if not text_report:
            raise ValueError("No assistant text found in messages.")
    else:
        text_report = str(text_or_messages)

    if not text_report:
        raise ValueError("Empty report text.")

    system_prompt = "You convert plaintext reports into full clean HTML documents."
    user_prompt = (
        "You are an expert technical writing assistant. "
        "Convert the following plaintext research report into a clean, structured HTML document. "
        "Include section headers, well-formatted paragraphs, inline links, and a clean readable layout. "
        "Ensure that all URLs are clickable and citation style is preserved.\n\n"
        "Respond ONLY with valid HTML (no explanation).\n\n"
        f"Report:\n{text_report}"
    )

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.5,
    )
    return resp.choices[0].message.content.strip()


### 🚀 End-to-End Pipeline

Run this cell to execute the full workflow:

1. Generate a research report (tools).
2. Reflect on the report.
3. Convert the report to HTML.

> You should see the rendered HTML below and two concise reflections in the console.

In [16]:
# 1) Research with tools
prompt_ = "Radio observations of recurrent novae"
preliminary_report = generate_research_report_with_tools(prompt_)
print("=== Research Report (preliminary) ===\n")
print(preliminary_report)

# 2) Reflection on the report (use the final TEXT to avoid ambiguity)
reflection_text = reflection_and_rewrite(preliminary_report)   # <-- pass text, not messages
print("=== Reflection on Report ===\n")
print(reflection_text['reflection'], "\n")
print("=== Revised Report ===\n")
print(reflection_text['revised_report'], "\n")


# 3) Convert the report to HTML (use the TEXT and correct function name)
html = convert_report_to_html(reflection_text['revised_report'])

print("=== Generated HTML (preview) ===\n")
print((html or "")[:600], "\n... [truncated]\n")

# 4) Display full HTML
display(HTML(html))


🛠️ arxiv_search_tool({'query': 'radio observations of recurrent novae', 'max_results': 5})
✅ Final answer:
### Radio Observations of Recurrent Novae

Radio observations are a crucial tool in understanding the dynamic processes and mass ejections involved in novae events, including recurrent novae. Here, I present a summary of recent findings and studies focused on these phenomena.

#### Overview

Recurrent novae are a subclass of cataclysmic variable stars that undergo repeated nova eruptions. These eruptions are caused by the thermonuclear runaway of hydrogen on the surface of a white dwarf star in a binary system, leading to the expulsion of material. Radio observations help trace the ejected material, studying its density, velocity, and composition over time.

#### Key Studies

1. **Lessons from Recurrent Novae YY Dor and Nova LMC 2009**:
   - **Study by Elena Mason and Frederick M. Walters**: This research discussed spectral characteristics and evolution in novae, highlighting comm

---

## ✅ Wrap-Up

We built a mini research agent that can:
- 🔎 call tools (arXiv + Tavily),
- 🧠 reflect on its own output,
- 📰 publish a clean HTML report.

Great job!

### Troubleshooting (quick)
- **Model/tool-call loop stalls?** Lower `MAX_TURNS` or print intermediate messages.
- **HTML looks odd?** Re-run conversion with a fresh assistant response.

